<a target="_blank" href="https://colab.research.google.com/github/genomicsxai/alphagenome-pytorch/blob/main/examples/notebooks/alphagenome_pytorch_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebooks shows how to run a forward pass using the PyTorch implementation of the AlphaGenome model.

You need to have model weights, track means, and the `alphagenome_pytorch` package in order to run it.

In order to convert JAX weights to PyTorch and extract track means, you can use utility scripts in the `alphagenome_pytorch` repository:

```py
python src/convert_weights.py ~/.cache/kagglehub/models/google/alphagenome/jax/all_folds/1 --output model.pth
python scripts/extract_track_means.py --output track_means.pt
```

In [ ]:
%pip install /content/alphagenome-pytorch

We will also use the reference implementation to compare outputs and visualize the results.

# Construct input data

In [6]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import jax
import jax.numpy as jnp

In [ ]:
import sys, os, site
print("python:", sys.executable)
print("sys.prefix:", sys.prefix)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("PATH head:", os.environ.get("PATH","").split(":")[:3])
print("site-packages:", site.getsitepackages()[:2])

In [ ]:
from alphagenome_research.model import dna_model

dev = jax.devices()[0]          # second GPU
model = dna_model.create(
    checkpoint_path="/path/to/jax-checkpoint",
    device=dev
)

In [13]:
from alphagenome.data import genome

interval = genome.Interval(chromosome='chr22', start=35677410, end=36725986)

organism = dna_model.Organism.HOMO_SAPIENS

extractor = model._get_fasta_extractor(organism)
ref_seq_str = extractor.extract(interval)
print(f"Sequence length: {len(ref_seq_str)}")

Sequence length: 1048576


In [14]:
import numpy as np

encoder = model._one_hot_encoder

ref_one_hot = encoder.encode(ref_seq_str) # (S, 4)
ref_batch = ref_one_hot[np.newaxis]       # (1, S, 4)

In [15]:
# Prepare auxiliary inputs required by _predict
organism_index = jnp.array([0], dtype=jnp.int32) # 0 for Human
track_metadata = model._metadata[organism]
strand_reindexing = jax.device_put(track_metadata.strand_reindexing)
negative_strand_mask = jnp.array([interval.negative_strand], dtype=bool) # False here

# PyTorch


In [ ]:
TORCH_TRACK_MEANS_PATH = '../../track_means.pt'
TORCH_WEIGHTS_PATH = '../../alphagenome_pytorch.pt'

In [19]:
import torch
from alphagenome_pytorch.model import AlphaGenome

track_means_dict = torch.load(TORCH_TRACK_MEANS_PATH)

In [20]:
pt_model = AlphaGenome(num_organisms=2,
                       track_means_dict=track_means_dict)
pt_model.load_state_dict(torch.load(TORCH_WEIGHTS_PATH), strict=False)
pt_model.eval()
pt_model.cuda();

In [21]:
pt_input = torch.tensor(ref_batch).to('cuda')
pt_org_index = torch.tensor(np.array([0])).to('cuda')

Forward pass:

In [22]:
with torch.no_grad():
    pt_out = pt_model(pt_input, pt_org_index)

Check the output:

In [23]:
pt_out['atac'][1]

tensor([[[0.0008, 0.0016, 0.0007,  ...,    nan,    nan,    nan],
         [0.0003, 0.0006, 0.0002,  ...,    nan,    nan,    nan],
         [0.0417, 0.0717, 0.0280,  ...,    nan,    nan,    nan],
         ...,
         [0.0460, 0.0695, 0.0276,  ...,    nan,    nan,    nan],
         [0.0092, 0.0153, 0.0059,  ...,    nan,    nan,    nan],
         [0.0091, 0.0148, 0.0060,  ...,    nan,    nan,    nan]]],
       device='cuda:0')

In [24]:
print(f"Available output types: {', '.join(list(pt_out.keys()))}")

Available output types: atac, dnase, procap, cage, rna_seq, chip_tf, chip_histone, pair_activations


Make track masks for plotting:

In [25]:
from alphagenome_research.model.metadata import metadata as metadata_lib
from alphagenome.models import dna_output

# Load metadata
metadata = metadata_lib.load(dna_model.Organism.HOMO_SAPIENS)

# Create mask
track_masks = metadata_lib.create_track_masks(
    metadata,
    requested_outputs={dna_output.OutputType.RNA_SEQ},
    requested_ontologies=[
        dna_output.ontology.from_curie('UBERON:0001157')
    ],
)

# Apply the mask
rna_seq_mask = track_masks[dna_output.OutputType.RNA_SEQ]

In [31]:
filtered_output = pt_out['rna_seq'][1].detach().float().cpu().numpy()[..., rna_seq_mask]  # (1, 1048576, num_matching_tracks)
# Get corresponding metadata for TrackData
track_metadata_df = metadata.rna_seq[rna_seq_mask]
from alphagenome.data import track_data
# Create a TrackData object
rna_seq_track = track_data.TrackData(
    values=filtered_output[0],
    resolution=1,
    metadata=track_metadata_df,
    interval=interval,
)

# JAX implementation

In [27]:
ref_device = jax.device_put(ref_batch)

print("Running Ref prediction...")
ref_preds = model._predict(
    model._params,
    model._state,
    ref_device,
    organism_index,
    negative_strand_mask=negative_strand_mask,
    strand_reindexing=strand_reindexing
)

Running Ref prediction...


# Comparison

In [28]:
from alphagenome.models import dna_output
from alphagenome.visualization import plot_components
from alphagenome.data import track_data
import pandas as pd
import matplotlib.pyplot as plt

In [29]:
jax_rna_seq = ref_preds[dna_output.OutputType.RNA_SEQ][0][:,rna_seq_mask]

jax_rna_seq_track = track_data.TrackData(
    values=jax_rna_seq,
    resolution=1,
    metadata=track_metadata_df,
    interval=interval,
)

In [ ]:
plot_components.plot(
    [
        plot_components.OverlaidTracks(
            tdata={
                'JAX_REF': jax_rna_seq_track,
                'TORCH': rna_seq_track,
            },
            colors={'JAX_REF': 'dimgrey', 'TORCH': 'red'},
        ),
    ],
    interval=interval.resize(2**15),
)
plt.show()